<a href="https://colab.research.google.com/github/safcleo12/python-warmup/blob/main/tutorials/Tutorial4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 3: Word Embeddings and PyTorch
This is the third tutorial with practical exercises for the lecture Introduction to Computational Linguistics in the Winter Semester 2026. Hands-on exercises are marked with 👋 ⚒ and questions are marked with ❓. Remember to first **store this notebook** in your Drive or GitHub.

---

## **Part 1: Already Trained Word Embeddings**

For faster processing, we are going to use GloVe embeddings, a close relative to word2vec from the lecture, with a dimensionality of 50 instead of 300 and a closer relation to count-based approaches. GloVe first counts how often every pair co-occurs in the whole corpus, then fits vectors to those counts. The results of word2vec and GloVe look similar. This is the original GloVe paper:


*   Pennington, J., Socher, R., & Manning, C. D. (2014). [Glove: Global vectors for word representation](https://aclanthology.org/D14-1162.pdf). In: Proceedings of the 2014 conference on empirical methods in natural language processing (EMNLP) (pp. 1532-1543).


There are several ways to train and use trained embeddings:
- `gensim`: simplest and straightforward implementation of `word2vec` and `GloVe`
- Training based on deep learning packages e.g., `tensorflow` and `torch`
- `spacy`: comes with a pre-trained embeddings models, using `GloVe`

Let us start with installing the gensim environment to handle different types of word embeddings and with some basic experiments.


In [ ]:
!pip install gensim

With the API of gensim we can load different already trained embeddings into our notebook environment. A list of available embedding models can be found [here](https://github.com/piskvorky/gensim-data).

In [ ]:
import gensim.downloader as api
from gensim.models import Word2Vec

glove_embeddings = api.load("glove-wiki-gigaword-50")

We can check the length of the entire vocabulary in the embedding model and access individual embeddings.

In [ ]:
print(f"Length of the vocabulary {len(glove_embeddings.key_to_index)}")
print(f"Embedding for the word good: \n {glove_embeddings["good"]}")

👋 ⚒ Exercise 1: Print how many dimensions a single vector in the GloVe embeddings has.

In [ ]:
# Your code here

The method `most_similar` allows us to explore the `topn`words that are most closely related to the word we input in vector space.

In [ ]:
glove_embeddings.most_similar("coffee", topn=8)

👋 ⚒ Exercise 2: Query the closest neighbors of three words of your own choice using `most_similar` and return only the top 5 results. Were any of the results surprising?  

In [ ]:
# Your examples and code here

We can also check the actual numbers that were learned for these words, which in this case are represented as numpy arrays.

In [ ]:
coffee = glove_embeddings["coffee"]
print(f"King as numpy size: {coffee.size}")
print(f"King as numpy array: {coffee}")
print(f"King really is a : {type(coffee)}")

👋 ⚒ Exercise 3: Similarity in this case means cosine similarity, which is the dot product of vectors u and v divided by the product of their magnitude (lengths). This division serves the purpose to strip away the magnitude and focus on the angle between the two vectors alone (=their distance). In numpy, there is the function `np.dot()`for the dot product of two 1D arrays and in the linear algebra (linalg) module a function for magnitude `np.linalg.norm()` that takes a single 1D array as input. Implement the `numpy_cosine` function.

$$ \cos(\theta) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\| \|\mathbf{v}\|} = \frac{\sum_{i=1}^{n} u_i v_i}{\sqrt{\sum_{i=1}^{n} u_i^2} \sqrt{\sum_{i=1}^{n} v_i^2}}$$



In [ ]:
import numpy as np

def numpy_cosine(u, v):
  #Your code here

print(numpy_cosine(glove_embeddings["king"], glove_embeddings["queen"]))
print(numpy_cosine(glove_embeddings["king"], glove_embeddings["banana"]))
print(glove_embeddings.similarity("king", "queen")) # this should match your result

This function can also be used for analogies, as the famous king is to queen as man is to woman analogy.

In [ ]:
glove_embeddings.most_similar(positive=["king", "woman"], negative=["man"], topn=3)

We can use matplotlib to visualize the proximity of words in vector space.

In [ ]:
from sklearn.decomposition import PCA
from matplotlib import pyplot as plt

def display_pca_scatterplot(model, words):

    word_vectors = np.array([model[w] for w in words])

    twodim = PCA().fit_transform(word_vectors)[:,:2]

    plt.figure(figsize=(6,6))
    plt.scatter(twodim[:,0], twodim[:,1], edgecolors='k', c='r')
    for word, (x,y) in zip(words, twodim):
        plt.text(x+0.05, y+0.05, word)

display_pca_scatterplot(glove_embeddings,
                        ['coffee', 'tea', 'beer', 'wine', 'water',
                         'hamburger', 'pizza',  'sushi', 'meatballs',
                         'dog', 'horse', 'cat', 'monkey', 'parrot', 'lizard',
                         'france', 'germany', 'hungary',
                         'school', 'college', 'university', 'institute'])

---

## **Part 2: PyTorch Basics**

To write code for neural networks from scratch instead of using already existing models and libraries, there are mainly two machine learning frameworks:

*   TensorFlow
*   Torch

Since these are high level core libraries in other programming languages, e.g. Torch is originally written in [Lua](https://www.lua.org/about.html), it is easier to use frameworks that build on top of them, adding usability and documentation. We are going to focus on Torch and its Python version [PyTorch](https://pytorch.org/), equally called torch in the Python libraries.

The most basic and important concept in PyTorch is that of a **Tensor**. [What is a tensor?](https://en.wikipedia.org/wiki/Tensor). A tensor is a grid of numbers: a list (1D), a table (2D), and so on. It is similar a Python list that is fast, supports maths on all elements at once, and can compute gradients.

While very similar to numpy arrays, tensors tend to speed up computations and offer more flexibility.

In [ ]:
import torch
import numpy

# Seed for random number generator to ensure reproducibility of
# random initializations
torch.manual_seed(1)

#Difference between tensor and numpy array
a = torch.ones(5)
print(f"Tensor: {a}")
print(f"Type of a (py)torch tensor: {type(a)}")
print(f"Numpy array: {a.numpy()}")
print(f"Type of a numpy tensor: {type(a.numpy())}")

Tensor: tensor([1., 1., 1., 1., 1.])
Type of a (py)torch tensor: <class 'torch.Tensor'>
Numpy array: [1. 1. 1. 1. 1.]
Type of a numpy tensor: <class 'numpy.ndarray'>


There are different ways to initialize a tensor exemplified in the following code cell.

In [ ]:
# This creates a randomly initialized 5 x 3 matrix
random_tensor = torch.rand(5,3)
print(f"Randomly initialized tensor: \n {random_tensor}")
print(f"The shape of the matrix is its dimensions (here 5x3): {random_tensor.shape}")

# This creates a 5 x 3 matrix filled with zeros and of dtype long
# There are eight datatypes in tensor, this one is a datatype of 64-bit integer (signed)
# Here are the others: https://pytorch.org/docs/stable/tensors.html
zeros = torch.zeros(4,3, dtype=torch.long)
print(f"Tensor initialized with zeros: \n {zeros}")

# Directly initialize a tensor with data
data = torch.tensor([[5.5, 3], [1, 2]])
print(f"Tensor initilialized with data: \n {data}")

Randomly initialized tensor: 
 tensor([[0.6607, 0.1251, 0.6004],
        [0.6201, 0.1652, 0.2628],
        [0.6705, 0.5896, 0.2873],
        [0.3486, 0.9579, 0.4075],
        [0.7819, 0.7165, 0.1768]])
The shape of the matrix is its dimensions (here 5x3): torch.Size([5, 3])
Tensor initialized with zeros: 
 tensor([[0, 0, 0],
        [0, 0, 0],
        [0, 0, 0],
        [0, 0, 0]])
Tensor initilialized with data: 
 tensor([[5.5000, 3.0000],
        [1.0000, 2.0000]])


A randomly initiatilzed tensor with only ones (`new_ones`), zeros (`torch.zeros`) or truly random (`torch.rand`), can be re-initialized with `torch.rand_like`. The difference is that with the other functions you need to pass the specific dimensions, whereas with `torch.rand_like` the tensor inherits these properties from the given tensor.    

In [ ]:
ones_only = rand.new_ones(5, 3, dtype=torch.float)
print(f"Randomly initialized tensor with ones only: \n {ones_only}")
print(f"And this is its size: {ones_only.size()}")

redefined_too = torch.rand_like(ones_only)
print(f"Redefining the ones only tensor: \n {redefined_too}")
print(f"And this is its size: {redefined_too.size()}")

Tensors in torch also support basic operations:

In [ ]:
a = torch.tensor([1., 2., 3.])
b = torch.tensor([4., 5., 6.])
print(a + b)
print(a * b)
print((a * b).sum())    # dot product = multiply, then sum
print(torch.dot(a, b))  # there is also a built in function for the dot product of two 1D tensors
print(a.norm())         # to get the magnitude (lengths) of a vector
print(a.shape)

Tensors can also be indexed and the important dot product can be easily calculated:

In [ ]:
c = torch.rand(5,3)
print(f"Entire tensor c: \n {c}")
print(f"c at row 2: {c[2]}")
print(f"c at row 0 and 2: \n {c[[0, 2]]}")
print(f"Element at index 1 of each row of c: {c[:, 1]}")
print(f"Dot product of every row of c with a: \n {c @ a}")

👋 ⚒ Exercise 4: Implement the function for the cosine similiarty in PyTorch in the code cell below. It is again the dot product of u and v divided by the product of their magnitude.

In [ ]:
def pytorch_cosine(u, v):
  #Your code here

print(pytorch_cosine(torch.tensor(glove_embeddings["king"]), torch.tensor(glove_embeddings["queen"])))
print(pytorch_cosine(torch.tensor(glove_embeddings["king"]), torch.tensor(glove_embeddings["banana"])))

Resizing: if you wish to change the shape of the tensor you can use torch.view:

In [ ]:
# Exercise: resizing: what effect do the following
# resize operations have on the tensors
x = torch.randn(4, 4)
y = x.view(16)
z = x.view(-1, 8)
print("Resizing: ")
print("Original")
print(x)
print("Resized view(16)", y, "\n")
print("Resized view(-1, 8)", z, "\n")


One of the reasons why we use **tensors** is *vectorized operations*, that is, operations that be conducted in parallel over a particular dimension of a tensor.

In [ ]:
data = torch.arange(1, 36, dtype=torch.float32).reshape(5, 7)
print("Data is:", data)

print("Taking the sum over rows:")
print(data.sum(dim=1)) #(5,)

print("Taking the sum over columns:")
print(data.sum(dim=0)) #(7,)

print("Taking the standard deviation over rows:")
print(data.std(dim=1))

👋 ⚒ Exercise 5: Write code that creates a `torch.tensor` with the following contents:
$\begin{bmatrix} 1 & 2.2 & 9.6 \\ 4 & -7.2 & 6.3 \end{bmatrix}$

Now compute the average of each row `.mean()` and each column.

What's the shape of the resulting tensor?

In [ ]:
# Your code here
my_first_tensor =

my_first_tensor_mean_rows =
my_first_tensor_mean_columns =

---

## **Part 3: Training a Neural Network with PyTorch**

The following examples are truly toy examples and require more and a more refined code for actual implementations.

In PyTorch, it is also possible to create your own neural network using the module nn. The Embedding layer below is the same as our variable `random_tensor` above. And the steps we perform below are also the same steps for training a neural network, only with more parameters and a more complicated architecture.

In [ ]:
import torch
import torch.nn as nn

# Again random number generator to ensure reproducibility
torch.manual_seed(1)

# Also a 5 x 3 matrix
emb = nn.Embedding(5, 3)

# We can also make these explicit
#emb = nn.Embedding(num_embeddings=5, embedding_dim=3)

# Out put the weights of the layer
print(emb.weight)

# Look up rows 0 and 2
print(emb(torch.tensor([0, 2])))

👋 ⚒ Exercise 5: We will perform the smallest possible learning probelm by finding a single parameter `w` so that `w * x` is close to `y`. The following for-loop simulates training steps by calculating the loss, performing backpropagation, and updating the weights.

Your task is to implement the loss function and keep track of the loss at each individual step with the losses list to see if the loss gets lower as expected during training.


In [ ]:
x = torch.tensor([1., 2., 3., 4., 5.])
y = torch.tensor([3.1, 5.9, 9.2, 11.8, 15.1])

# The variable requires_grad states whether you wish to track this for gradients or not
w = torch.tensor(0.0, requires_grad=True)

# The learning rate = step size
lr = 0.01
losses = []
# We repeat the training step 50 times
for step in range(50):
  prediction = w * x

  # This loss function is called the Mean Squared Error (MSE), because it takes the mean of (prediction - y) squared
  loss = # Your code here

  # We need to compute the gradients by means of backpropagation
  loss.backward()

  # with torch.no_grad(): do not record operations in this block for calculating the gradients
  # We only want to trace steps towards the prediction for computing the gradients
  # The following lines update/change the parameters based on the calcualted gradients in the backward step (optimization)
  with torch.no_grad():
      w -= lr * w.grad
      w.grad.zero_()

  # Your code here to store the loss.item() in the losses


print(w)
print(losses)
plt.plot(losses); plt.xlabel("step"); plt.ylabel("loss")


We can perform the same steps with the optmizer already provided by PyTorch.

👋 ⚒ Exercise 6: Change the learning rate in the following code cell to 0.1. What happens?

In [ ]:
w = torch.tensor(0.0, requires_grad=True)

# SGD means Stochastic Gradient Descent
optim = torch.optim.SGD([w], lr=0.01)
losses = []
for step in range(50):
    loss = ((w * x - y) ** 2).mean()
    optim.zero_grad()      # forget old gradients
    loss.backward()      # compute new gradients
    optim.step()           # update parameters
    losses.append(loss.item())
plt.plot(losses); print(w)